Reinforcement learning (RL) has rapidly become a core stage of modern foundation-model development. While large-scale pretraining remains essential, today's most capable models rely heavily on post-training techniques to improve reasoning, tool use, and multi-turn interaction. These workflows depend on scalable reinforcement learning infrastructure capable of running across multi-node GPU clusters.

This notebook explains what verl is, why RL workloads are a strong fit for AMD GPUs, what features are already supported today, and how to run an end-to-end RL training workflow on ROCm.

# Introducing verl

verl is a flexible, efficient and production-ready RL training library for large language models (LLMs).

verl is flexible and easy to use with:

- **Easy extension of diverse RL algorithms**: The hybrid-controller programming model enables flexible representation and efficient execution of complex post-training dataflows. Build RL dataflows such as GRPO, PPO in a few lines of code.

- **Seamless integration of existing LLM infra with modular APIs**: Decouples computation and data dependencies, enabling seamless integration with existing LLM frameworks, such as FSDP, Megatron-LM, vLLM, SGLang, etc

- **Flexible device mapping**: Supports various placement of models onto different sets of GPUs for efficient resource utilization and scalability across different cluster sizes.

- Ready integration with popular HuggingFace models

# Why RL Workloads Fit AMD Instinct GPUs 

Reinforcement learning workloads differ from pretraining in a crucial way: rollout generation dominates compute.

Modern RL training may spend 70–90% of GPU time generating long sequences across thousands of parallel environments. This makes memory capacity and bandwidth critical performance factors.

AMD Instinct MI GPUs provide:

Large HBM memory capacity
High memory bandwidth
Efficient long-context inference

These properties can mitigate the classical rollout-heavy bottleneck within RL pipelines.

# Getting Started: Run verl on AMD GPUs

#### Install verl and download assets

In [ ]:
!git clone https://github.com/verl-project/verl-omni.git
%cd verl
!git checkout 0eb50ec4
!pip install -e .

Download the example model and datasets:

In [ ]:
# Prepare dataset:
python3 examples/data_preprocess/gsm8k.py --local_dir ../data/gsm8k

In [ ]:
# Download model:
python3 -c "import transformers; transformers.AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B')"

#### Configure settings

In [3]:
MODEL_PATH="Qwen/Qwen3-4B"
train_files="../data/gsm8k/train.parquet"
test_files="../data/gsm8k/test.parquet"

#### Set environment variables

In [5]:
!export HIP_VISIBLE_DEVICES=0
!GPUS_PER_NODE=1

#### Launch RL training

In [ ]:
MODEL_PATH="Qwen/Qwen3-4B"
GPU_MEMORY_UTILIZATION=0.5
max_token_len_per_gpu=24576

# Scale batch sizes and TP based on GPU count
if [ "$GPUS_PER_NODE" -eq 1 ]; then
    TP_VALUE=1
    TRAIN_BATCH_SIZE=16
    MINI_BATCH_SIZE=16
elif [ "$GPUS_PER_NODE" -le 2 ]; then
    TP_VALUE=2
    TRAIN_BATCH_SIZE=32
    MINI_BATCH_SIZE=16
else
    TP_VALUE=2
    TRAIN_BATCH_SIZE=128
    MINI_BATCH_SIZE=64
fi

# ── Launch GRPO LoRA-merge training ──────────────────────────────────────────
python3 -m verl.trainer.main_ppo \
    algorithm.adv_estimator=grpo \
    trainer.val_before_train=True \
    data.train_files=$train_files \
    data.val_files=$test_files \
    data.train_batch_size=$TRAIN_BATCH_SIZE \
    data.max_prompt_length=1024 \
    data.max_response_length=1024 \
    data.filter_overlong_prompts=True \
    data.truncation='error' \
    data.shuffle=False \
    actor_rollout_ref.model.path=$MODEL_PATH \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.model.lora_rank=32 \
    actor_rollout_ref.model.lora_alpha=64 \
    actor_rollout_ref.actor.optim.lr=1.0e-05 \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_mini_batch_size=$MINI_BATCH_SIZE \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${max_token_len_per_gpu} \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_coef=0.001 \
    actor_rollout_ref.actor.kl_loss_type=low_var_kl \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.actor.strategy=fsdp2 \
    actor_rollout_ref.actor.fsdp_config.model_dtype=bf16 \
    actor_rollout_ref.actor.fsdp_config.param_offload=False \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=False \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
    actor_rollout_ref.rollout.tensor_model_parallel_size=$TP_VALUE \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.gpu_memory_utilization=$GPU_MEMORY_UTILIZATION \
    actor_rollout_ref.rollout.n=8 \
    actor_rollout_ref.rollout.load_format=safetensors \
    actor_rollout_ref.rollout.layered_summon=True \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    actor_rollout_ref.ref.strategy=fsdp2 \
    actor_rollout_ref.ref.fsdp_config.model_dtype=bf16 \
    algorithm.use_kl_in_reward=False \
    trainer.use_legacy_worker_impl=disable \
    trainer.critic_warmup=0 \
    trainer.logger='["console","tensorboard"]' \
    trainer.project_name='grpo_qwen_llm' \
    trainer.experiment_name='=qwen3_4b_grpo_lora_rocm' \
    trainer.n_gpus_per_node=8 \
    trainer.nnodes=1 \
    trainer.save_freq=-1 \
    trainer.test_freq=10 \
    trainer.total_epochs=50

This launches a full RL pipeline:

- Ray cluster initialization

- vLLM rollout workers

- GRPO training loop

- On-policy rollout → train → update cycle